In [1]:
%load_ext autoreload
%autoreload 2
import sys
sys.path.append(".")
print("Ready")

Ready


In [ ]:
%%writefile tools/llm_clients.py
import os
import time
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from google import genai
from google.genai import types
from google.genai import errors as genai_errors
from tavily import TavilyClient

load_dotenv()

GROQ_KEY = os.getenv("GROQ_API_KEY")
GEMINI_KEY = os.getenv("GEMINI_API_KEY")
TAVILY_KEY = os.getenv("TAVILY_API_KEY")

assert GROQ_KEY, "Missing GROQ_API_KEY"
assert GEMINI_KEY, "Missing GEMINI_API_KEY"
assert TAVILY_KEY, "Missing TAVILY_API_KEY"

reasoning_llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0)
gemini_client = genai.Client(api_key=GEMINI_KEY)
tavily_client = TavilyClient(api_key=TAVILY_KEY)


def is_rate_limit_error(e: Exception) -> bool:
    """Shared across every agent now (moved here from web_search_agent on
    Day 18) — detects rate-limit errors by message content since exception
    TYPES differ between Groq/Gemini/Tavily SDKs, but the signal (429,
    'rate limit', 'RESOURCE_EXHAUSTED') is consistent enough to check as text."""
    msg = str(e).lower()
    return "429" in msg or "rate limit" in msg or "resource_exhausted" in msg


def call_llm(prompt: str, max_retries: int = 2) -> str:
    """Shared LLM call wrapper, used by every agent. As of Day 18, retries
    automatically on rate-limit errors with exponential backoff — previously
    this retry logic only existed in web_search_agent, which meant Day 17's
    real run showed summarization and report generation failing on 429s with
    no recovery. Now every agent gets this for free by importing call_llm."""
    last_error = None
    for attempt in range(max_retries + 1):
        try:
            return reasoning_llm.invoke(prompt).content
        except Exception as e:
            last_error = e
            if is_rate_limit_error(e) and attempt < max_retries:
                wait = 2 ** attempt
                time.sleep(wait)
                continue
            raise
    raise last_error


def research_with_fallback(query: str) -> dict:
    """Tries Gemini (grounded search + citations) first. Falls back to Tavily
    on Gemini-specific failures (its own API/client/server errors), not on
    ANY exception — narrowed on Day 18 per the gap flagged back on Day 5, so
    a real bug in this function's own code can't get silently misattributed
    to 'Gemini is down' and masked by the Tavily fallback."""
    try:
        resp = gemini_client.models.generate_content(
            model="gemini-3.6-flash",
            contents=query,
            config=types.GenerateContentConfig(tools=[types.Tool(google_search=types.GoogleSearch())])
        )
        sources = []
        if resp.candidates[0].grounding_metadata:
            sources = [
                {"title": c.web.title, "url": c.web.uri}
                for c in resp.candidates[0].grounding_metadata.grounding_chunks
            ]
        return {"answer": resp.text, "sources": sources, "provider": "gemini"}
    except (genai_errors.ClientError, genai_errors.ServerError) as e:
        print(f" Gemini unavailable ({type(e).__name__}), falling back to Tavily...")
        results = tavily_client.search(query=query, max_results=3)
        sources = [{"title": r["title"], "url": r["url"]} for r in results["results"]]
        answer = " ".join(r.get("content", "")[:300] for r in results["results"])
        return {"answer": answer, "sources": sources, "provider": "tavily"}

In [1]:
import os
print(os.getcwd())

c:\Users\RIDDHI ASHAR\Desktop\multi-agent-research-system


In [2]:
import os
print("Current directory:", os.getcwd())

Current directory: c:\Users\RIDDHI ASHAR\Desktop\multi-agent-research-system


In [3]:
%%writefile tools/llm_clients.py
import os
import time
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from google import genai
from google.genai import types
from google.genai import errors as genai_errors
from tavily import TavilyClient

load_dotenv()

GROQ_KEY = os.getenv("GROQ_API_KEY")
GEMINI_KEY = os.getenv("GEMINI_API_KEY")
TAVILY_KEY = os.getenv("TAVILY_API_KEY")

assert GROQ_KEY, "Missing GROQ_API_KEY"
assert GEMINI_KEY, "Missing GEMINI_API_KEY"
assert TAVILY_KEY, "Missing TAVILY_API_KEY"

reasoning_llm = ChatGroq(model="qwen/qwen3.8-27b", temperature=0)
gemini_client = genai.Client(api_key=GEMINI_KEY)
tavily_client = TavilyClient(api_key=TAVILY_KEY)


def is_rate_limit_error(e: Exception) -> bool:
    """Shared across every agent now (moved here from web_search_agent on
    Day 18) — detects rate-limit errors by message content since exception
    TYPES differ between Groq/Gemini/Tavily SDKs, but the signal (429,
    'rate limit', 'RESOURCE_EXHAUSTED') is consistent enough to check as text."""
    msg = str(e).lower()
    return "429" in msg or "rate limit" in msg or "resource_exhausted" in msg


def call_llm(prompt: str, max_retries: int = 2) -> str:
    """Shared LLM call wrapper, used by every agent. As of Day 18, retries
    automatically on rate-limit errors with exponential backoff — previously
    this retry logic only existed in web_search_agent, which meant Day 17's
    real run showed summarization and report generation failing on 429s with
    no recovery. Now every agent gets this for free by importing call_llm."""
    last_error = None
    for attempt in range(max_retries + 1):
        try:
            return reasoning_llm.invoke(prompt).content
        except Exception as e:
            last_error = e
            if is_rate_limit_error(e) and attempt < max_retries:
                wait = 2 ** attempt
                time.sleep(wait)
                continue
            raise
    raise last_error


def research_with_fallback(query: str) -> dict:
    """Tries Gemini (grounded search + citations) first. Falls back to Tavily
    on Gemini-specific failures (its own API/client/server errors), not on
    ANY exception — narrowed on Day 18 per the gap flagged back on Day 5, so
    a real bug in this function's own code can't get silently misattributed
    to 'Gemini is down' and masked by the Tavily fallback."""
    try:
        resp = gemini_client.models.generate_content(
            model="gemini-3.6-flash",
            contents=query,
            config=types.GenerateContentConfig(tools=[types.Tool(google_search=types.GoogleSearch())])
        )
        sources = []
        if resp.candidates[0].grounding_metadata:
            sources = [
                {"title": c.web.title, "url": c.web.uri}
                for c in resp.candidates[0].grounding_metadata.grounding_chunks
            ]
        return {"answer": resp.text, "sources": sources, "provider": "gemini"}
    except (genai_errors.ClientError, genai_errors.ServerError) as e:
        print(f"Gemini unavailable ({type(e).__name__}), falling back to Tavily...")
        results = tavily_client.search(query=query, max_results=3)
        sources = [{"title": r["title"], "url": r["url"]} for r in results["results"]]
        answer = " ".join(r.get("content", "")[:300] for r in results["results"])
        return {"answer": answer, "sources": sources, "provider": "tavily"}

Overwriting tools/llm_clients.py


In [6]:
import tools.llm_clients as llc

calls = {"count": 0}
original_llm = llc.reasoning_llm

class FakeResponse:
    content = "recovered successfully"

class FakeLLM:
    def invoke(self, prompt):
        calls["count"] += 1
        if calls["count"] < 2:
            raise Exception("429 rate_limit_exceeded: too many requests")
        return FakeResponse()

llc.reasoning_llm = FakeLLM()

result = llc.call_llm("test prompt")
print("Result:", result)
print("Attempts made:", calls["count"])
assert calls["count"] == 2
assert result == "recovered successfully"
print(" call_llm retried once on rate limit, then succeeded")

llc.reasoning_llm = original_llm  # restore

Result: recovered successfully
Attempts made: 2
 call_llm retried once on rate limit, then succeeded


In [8]:
calls2 = {"count": 0}

class AlwaysBrokenLLM:
    def invoke(self, prompt):
        calls2["count"] += 1
        raise ValueError("malformed request: missing required field")  # unrelated to rate limiting

llc.reasoning_llm = AlwaysBrokenLLM()

try:
    llc.call_llm("test prompt")
    print(" FAIL: should have raised")
except ValueError:
    print(f"Correctly raised immediately, no wasted retries (attempts: {calls2['count']})")
    assert calls2["count"] == 1

llc.reasoning_llm = original_llm  # restore

Correctly raised immediately, no wasted retries (attempts: 1)


In [9]:
%%writefile tests/test_llm_clients.py
import tools.llm_clients as llc


def test_is_rate_limit_error_detects_variants():
    assert llc.is_rate_limit_error(Exception("429 Too Many Requests"))
    assert llc.is_rate_limit_error(Exception("RESOURCE_EXHAUSTED: quota"))
    assert llc.is_rate_limit_error(Exception("rate limit reached"))
    assert not llc.is_rate_limit_error(Exception("connection reset"))
    assert not llc.is_rate_limit_error(Exception("invalid API key"))


def test_call_llm_retries_on_rate_limit_then_succeeds(monkeypatch):
    calls = {"count": 0}

    class FakeResponse:
        content = "ok"

    def flaky(prompt):
        calls["count"] += 1
        if calls["count"] < 2:
            raise Exception("429 rate limit")
        return FakeResponse()

    monkeypatch.setattr(llc.reasoning_llm, "invoke", flaky)
    monkeypatch.setattr(llc.time, "sleep", lambda s: None)

    result = llc.call_llm("prompt", max_retries=2)
    assert result == "ok"
    assert calls["count"] == 2


def test_call_llm_does_not_retry_non_rate_limit_errors(monkeypatch):
    calls = {"count": 0}

    def boom(prompt):
        calls["count"] += 1
        raise ValueError("real bug")

    monkeypatch.setattr(llc.reasoning_llm, "invoke", boom)

    try:
        llc.call_llm("prompt", max_retries=2)
        assert False, "should have raised"
    except ValueError:
        pass
    assert calls["count"] == 1


def test_call_llm_exhausts_retries_and_raises(monkeypatch):
    calls = {"count": 0}

    def always_limited(prompt):
        calls["count"] += 1
        raise Exception("429 rate limit")

    monkeypatch.setattr(llc.reasoning_llm, "invoke", always_limited)
    monkeypatch.setattr(llc.time, "sleep", lambda s: None)

    try:
        llc.call_llm("prompt", max_retries=2)
        assert False, "should have raised after exhausting retries"
    except Exception as e:
        assert "429" in str(e)
    assert calls["count"] == 3  # initial attempt + 2 retries

Writing tests/test_llm_clients.py


In [10]:
%%writefile tests/test_llm_clients.py
import tools.llm_clients as llc


class FakeResponse:
    content = "ok"


def test_is_rate_limit_error_detects_variants():
    assert llc.is_rate_limit_error(Exception("429 Too Many Requests"))
    assert llc.is_rate_limit_error(Exception("RESOURCE_EXHAUSTED: quota"))
    assert llc.is_rate_limit_error(Exception("rate limit reached"))
    assert not llc.is_rate_limit_error(Exception("connection reset"))
    assert not llc.is_rate_limit_error(Exception("invalid API key"))


def test_call_llm_retries_on_rate_limit_then_succeeds(monkeypatch):
    calls = {"count": 0}

    class FlakyLLM:
        def invoke(self, prompt):
            calls["count"] += 1
            if calls["count"] < 2:
                raise Exception("429 rate limit")
            return FakeResponse()

    monkeypatch.setattr(llc, "reasoning_llm", FlakyLLM())
    monkeypatch.setattr(llc.time, "sleep", lambda s: None)

    result = llc.call_llm("prompt", max_retries=2)
    assert result == "ok"
    assert calls["count"] == 2


def test_call_llm_does_not_retry_non_rate_limit_errors(monkeypatch):
    calls = {"count": 0}

    class BrokenLLM:
        def invoke(self, prompt):
            calls["count"] += 1
            raise ValueError("real bug")

    monkeypatch.setattr(llc, "reasoning_llm", BrokenLLM())

    try:
        llc.call_llm("prompt", max_retries=2)
        assert False, "should have raised"
    except ValueError:
        pass
    assert calls["count"] == 1


def test_call_llm_exhausts_retries_and_raises(monkeypatch):
    calls = {"count": 0}

    class AlwaysLimitedLLM:
        def invoke(self, prompt):
            calls["count"] += 1
            raise Exception("429 rate limit")

    monkeypatch.setattr(llc, "reasoning_llm", AlwaysLimitedLLM())
    monkeypatch.setattr(llc.time, "sleep", lambda s: None)

    try:
        llc.call_llm("prompt", max_retries=2)
        assert False, "should have raised after exhausting retries"
    except Exception as e:
        assert "429" in str(e)
    assert calls["count"] == 3

Overwriting tests/test_llm_clients.py


In [ ]:
from graph.build_graph import build_graph
import time

app = build_graph()

initial_state = {
    "query": "Compare EV battery strategies of Tesla, BYD, and CATL",
    "plan": [], "next_agent": "", "iteration_count": 0, "needs_more_research": False,
    "search_results": [], "extracted_facts": [], "fact_check_results": [], "summaries": [],
    "errors": [], "final_report": None, "model_usage_log": [],
}

start = time.time()
result = app.invoke(initial_state)
elapsed = time.time() - start

print(f"Total time: {elapsed:.2f}s | Iterations: {result['iteration_count']}")
print(f"Search results: {len(result['search_results'])} | Errors: {len(result['errors'])}")

summary_statuses = [s["status"] for s in result["summaries"]]
print(f"Summary statuses: {summary_statuses}")

method = [m for m in result["model_usage_log"] if m.get("agent") == "report_generator_agent"]
print(f"Report generation method: {method}")